In [1]:
import json
import pandas as pd
import glob
import requests

def merge_all_data() :

    batch_files = glob.glob("../data/raw/D1/Movies_Api_Respond_p*.json")

    merged_data = []

    for file_path in batch_files:
        with open(file_path, 'r') as file:
            batch_data = json.load(file)
            # tab = pd.DataFrame(batch_data['results'])
            merged_data.extend(batch_data['results'])


    with open("../data/All_Movies.json", "w") as file:
        json.dump(merged_data, file, indent=4)

    return merged_data

df = merge_all_data()
df = pd.DataFrame(df)

print(df.columns.tolist())

['adult', 'backdrop_path', 'genre_ids', 'id', 'title', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'release_date', 'softcore', 'video', 'vote_average', 'vote_count']


In [2]:
import json
import requests
from dotenv import load_dotenv
import os
import time


def get_api_key():
    load_dotenv()
    return os.getenv("TMDB_API_KEY")


def get_movie_details(id):
    API_KEY = get_api_key()

    params = {
        "api_key": API_KEY,
        "language": "en-US",
    }

    url = f"https://api.themoviedb.org/3/movie/{id}"

    data = None

    try:
        response = requests.get(url, params=params)

        if response.status_code == 200:
            print("Details success!")
            data = response.json()
        else:
            print("Details error:", response.status_code)
            print(response.text)

    except requests.exceptions.RequestException as e:
        print("Request failed:", e)

    return data


def get_movie_keywords(id):
    API_KEY = get_api_key()

    params = {
        "api_key": API_KEY,
        "language": "en-US",
    }

    url = f"https://api.themoviedb.org/3/movie/{id}/keywords"

    data = None

    try:
        response = requests.get(url, params=params)

        if response.status_code == 200:
            print("Keywords success!")
            data = response.json()
        else:
            print("Keywords error:", response.status_code)
            print(response.text)

    except requests.exceptions.RequestException as e:
        print("Request failed:", e)

    return data


def get_response_from_tmdb():

    with open("../data/All_Movies.json", "r") as file:
        All_Movies = json.load(file)

    All_Movies_List = []

    for movie in All_Movies:

        movie_id = movie["id"]

        details = get_movie_details(movie_id)
        keywords_data = get_movie_keywords(movie_id)


        if details is None or keywords_data is None:
            continue

        genres = [
            genre["name"]
            for genre in details["genres"]
        ]


        keywords = [
            keyword["name"]
            for keyword in keywords_data["keywords"]
        ]


        movie_data = {
            "movie_id": movie["id"],
            "title": movie["title"],
            "overview": movie["overview"],
            "release_date": movie["release_date"],
            "runtime": details["runtime"],
            "original_language": movie["original_language"],
            "genres": genres,
            "keywords": keywords,
            "budget": details["budget"],
            "revenue": details["revenue"],
            "popularity": movie["popularity"],
            "vote_average": movie["vote_average"],
            "vote_count": movie["vote_count"]
        }


        All_Movies_List.append(movie_data)

        time.sleep(0.2)

    return All_Movies_List


All_Movies_List = get_response_from_tmdb()


with open("../data/raw/D2/Clean_Movies_test.json", "w") as f:
    json.dump(All_Movies_List, f, indent=4)

Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Keywords success!
Details success!
Key

In [3]:
import json
import requests
from dotenv import load_dotenv
import os
import time
df = pd.read_json("../data/raw/D2/Clean_Movies_test.json")

print(df.head())
print(df.shape)
print(df.columns.tolist())
print(df.info())

   movie_id                      title  \
0    969681  Spider-Man: Brand New Day   
1   1423191              Resident Evil   
2   1101383      The End of Oak Street   
3   1204680            Coyote vs. Acme   
4   1368337                The Odyssey   

                                            overview release_date  runtime  \
0  Fighting crime full-time as Spider-Man in a wo...   2026-07-29      145   
1  Medical courier Bryan unwittingly finds himsel...   2026-09-16       95   
2  After a mysterious cosmic event rips Oak Stree...   2026-08-12      100   
3  After Acme products fail him one too many time...   2026-08-17      103   
4  Odysseus, the legendary King of Ithaca, embark...   2026-07-15      173   

  original_language                                genres  \
0                en  [Science Fiction, Action, Adventure]   
1                en  [Horror, Science Fiction, Adventure]   
2                en  [Science Fiction, Mystery, Thriller]   
3                en           [Com

In [4]:
print(df.isnull().sum())

movie_id             0
title                0
overview             0
release_date         0
runtime              0
original_language    0
genres               0
keywords             0
budget               0
revenue              0
popularity           0
vote_average         0
vote_count           0
dtype: int64


In [5]:
print(df[df["keywords"].isnull()][["title", "keywords"]].head())

Empty DataFrame
Columns: [title, keywords]
Index: []


In [6]:
print(df["movie_id"].duplicated().sum())

259


In [11]:
text_columns = [
    "title",
    "overview",
    "original_language"
]

for column in text_columns:

    empty_after_strip = (
        df[column]
        .astype("string")
        .str.strip()
        .eq("")
    )

    print(
        column,
        ":",
        empty_after_strip.sum()
    )

title : 0
overview : 9
original_language : 0


In [12]:
duplicates = df[
    df["movie_id"].duplicated(keep=False)
]

print(duplicates.sort_values("movie_id"))

df = df.drop_duplicates(subset=["movie_id"])
print(df.duplicated(subset=["movie_id"]).sum())

      movie_id                     title  \
1968        81      Warriors of the Wind   
2020        81      Warriors of the Wind   
320         96      Beverly Hills Cop II   
584         96      Beverly Hills Cop II   
316        147             The 400 Blows   
...        ...                       ...   
555    1584215            The Internship   
667    1729723             Yellow Mirror   
803    1729723             Yellow Mirror   
2966   1762506  Band of Brothers: Legacy   
2903   1762506  Band of Brothers: Legacy   

                                               overview release_date  runtime  \
1968  After a global war, the seaside kingdom known ...   1984-03-11       95   
2020  After a global war, the seaside kingdom known ...   1984-03-11       95   
320   Axel Foley returns to the land of sunshine and...   1987-05-18      103   
584   Axel Foley returns to the land of sunshine and...   1987-05-18      103   
316   For young Parisian boy Antoine Doinel, life is...   1959-06-

In [13]:
print(df.dtypes)

movie_id               int64
title                 object
overview              object
release_date          object
runtime                int64
original_language     object
genres                object
keywords              object
budget                 int64
revenue                int64
popularity           float64
vote_average         float64
vote_count             int64
dtype: object


In [15]:
df["release_date"] = pd.to_datetime(df["release_date"])

print(df["release_date"].head())
df["release_date"] = df["release_date"].astype(str)


0   2026-07-29
1   2026-09-16
2   2026-08-12
3   2026-08-17
4   2026-07-15
Name: release_date, dtype: datetime64[ns]


In [16]:
print(df.dtypes)

movie_id               int64
title                 object
overview              object
release_date          object
runtime                int64
original_language     object
genres                object
keywords              object
budget                 int64
revenue                int64
popularity           float64
vote_average         float64
vote_count             int64
dtype: object


In [20]:
int_columns = ["runtime", "budget", "revenue", "vote_count"]
float_columns = ["popularity", "vote_average"]
string_columns = ['title' , "overview" ,"original_language"]

df[int_columns] = df[int_columns].astype(int)
df[float_columns] = df[float_columns].astype(int)
df[string_columns] = df[string_columns].astype(str)

def ensure_list(value):
    if value is None:
        return []

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    return [value]


df["genres"] = df["genres"].apply(ensure_list)
df["keywords"] = df["keywords"].apply(ensure_list)

print(df.dtypes)

print(df["genres"].apply(type).value_counts())
print(df["keywords"].apply(type).value_counts())

print(df["release_date"].head())
df["release_date"] = df["release_date"].astype(str)



movie_id              int64
title                object
overview             object
release_date         object
runtime               int64
original_language    object
genres               object
keywords             object
budget                int64
revenue               int64
popularity            int64
vote_average          int64
vote_count            int64
dtype: object
genres
<class 'list'>    2740
Name: count, dtype: int64
keywords
<class 'list'>    2740
Name: count, dtype: int64
0    2026-07-29
1    2026-09-16
2    2026-08-12
3    2026-08-17
4    2026-07-15
Name: release_date, dtype: object


In [21]:
df.to_json(
    "../data/processed/Movies_Cleaned.json",
    orient="records",
    indent=4
)

In [22]:
from pymongo import MongoClient
import json

client = MongoClient("mongodb://tmdb_mongodb:27017/")

db = client["movie_db"]

collection = db["movies"]

with open(
    "../data/processed/Movies_Cleaned.json",
    "r",
    encoding="utf-8"
) as file:

    movies = json.load(file)
collection.delete_many({})
result = collection.insert_many(movies)

print("Inserted:", len(result.inserted_ids))
print("Total documents:", collection.count_documents({}))

print(collection.find_one())

Inserted: 2740
Total documents: 2740
{'_id': ObjectId('6ac39024d20681e669ace04f'), 'movie_id': 969681, 'title': 'Spider-Man: Brand New Day', 'overview': "Fighting crime full-time as Spider-Man in a world that doesn't remember him—and the pressure of seeing his old friends move on without him—sparks a change in Peter Parker he may not have the power to control. But that transformation might also be the only thing that can stop a shocking new threat to the city and those he loves - a powerful villain no one can even see.", 'release_date': '2026-07-29', 'runtime': 145, 'original_language': 'en', 'genres': ['Science Fiction', 'Action', 'Adventure'], 'keywords': ['mind control', 'new york city', 'hero', 'mutation', 'secret identity', 'superhero', 'spider', 'villain', 'based on comic', 'sequel', 'transhumanism', 'super power', 'masked vigilante', 'spider web', 'aftercreditsstinger', 'marvel cinematic universe (mcu)', 'masked superhero', 'fight for justice', 'genetic mutation'], 'budget': 225